# Apache Spark 4.1.1 — Scenarios & Vanilla vs Gluten/Velox Benchmark

This notebook has two parts:
- **Part A (cells 0–12):** Core Spark scenarios — DataFrame API, SQL, caching, Iceberg, Delta, Streaming, UDFs.
- **Part B (cells 13–18):** TPC-H style benchmark comparing Vanilla Spark vs Gluten/Velox.

| Service | URL |
|---|---|
| Spark Master UI | http://localhost:8080 |
| Worker 1 | http://localhost:8081 |
| Worker 2 | http://localhost:8082 |
| History Server | http://localhost:18080 |
| Spark App UI | http://localhost:4040 |

---
### Gluten/Velox compatibility note
Gluten 1.4.x (latest stable) officially supports **Spark 3.2–3.5** only.  
Spark 4.x support is planned for **Gluten 1.5** (not released as of April 2026).  
With `GLUTEN_ENABLED=true` on Spark 4.1.1 some operators may fall back to JVM — this is expected and safe.  
Check the physical plan for `Velox` prefixed operators to see what was actually offloaded.

---
# Part A — Core Spark Scenarios

## 0 — SparkSession

In [1]:
import os, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

# Gluten config is already written to spark-defaults.conf by entrypoint.sh.
# No --conf flags here — those crash the driver JVM when passed via SparkConf.
spark = (
    SparkSession.builder
    .appName(f'spark-{"gluten" if GLUTEN_ENABLED else "vanilla"}')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

mode = 'Gluten/Velox' if GLUTEN_ENABLED else 'Vanilla'
print(f'Spark {spark.version}  |  Mode: {mode}')

if GLUTEN_ENABLED:
    plugins = spark.conf.get('spark.plugins', '')
    print('Gluten plugin: LOADED' if 'GlutenPlugin' in plugins else
          'Gluten plugin: NOT loaded — check entrypoint logs and JAR path')

spark.range(3).show()


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 22:13:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/07 22:13:38 WARN Component: Excluding runtime-incompatible component: velox-paimon.
26/04/07 22:13:38 WARN Component: Excluding runtime-incompatible component: velox-hudi.
W20260407 22:13:40.078941   452 MemoryArbitrator.cpp:84] Query memory capacity[288.00MB] is set for NOOP arbitrator which has no capacity enforcement
26/04/07 22:13:41 WARN SparkShimProvider: Spark runtime version 3.5.8 is not matched with Gluten's fully tested version 3.5.5


Spark 3.5.8  |  Mode: Gluten/Velox
Gluten plugin: LOADED


26/04/07 22:13:44 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=0], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(id#0L, Some(Etc/UTC)), class name: ToPrettyString.
[Stage 0:>                                                          (0 + 2) / 2]

+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



## 1 — DataFrame API

In [2]:
schema = StructType([
    StructField('id',         IntegerType()),
    StructField('name',       StringType()),
    StructField('department', StringType()),
    StructField('salary',     DoubleType()),
    StructField('country',    StringType()),
])

data = [
    (1, 'Alice',  'Engineering', 4500.0, 'SK'),
    (2, 'Bob',    'Marketing',   3200.0, 'CZ'),
    (3, 'Carol',  'Engineering', 5100.0, 'SK'),
    (4, 'Dan',    'HR',          2900.0, 'HU'),
    (5, 'Eve',    'Engineering', 4800.0, 'CZ'),
    (6, 'Frank',  'Marketing',   3500.0, 'SK'),
    (7, 'Grace',  'HR',          3100.0, 'SK'),
    (8, 'Hank',   'Engineering', 5300.0, 'CZ'),
]

employees = spark.createDataFrame(data, schema)

employees \
    .filter(F.col('salary') > 3000) \
    .withColumn('salary_eur', F.round(F.col('salary') * 0.92, 2)) \
    .withColumn('level',
        F.when(F.col('salary') >= 5000, 'Senior')
         .when(F.col('salary') >= 4000, 'Mid')
         .otherwise('Junior')
    ).show()

26/04/07 22:13:49 WARN GlutenFallbackReporter: Validation failed for plan: Project[QueryId=1], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(id#7, Some(Etc/UTC)), class name: ToPrettyString.
[Stage 2:===================>                                       (1 + 2) / 3]

+---+-----+-----------+------+-------+----------+------+
| id| name| department|salary|country|salary_eur| level|
+---+-----+-----------+------+-------+----------+------+
|  1|Alice|Engineering|4500.0|     SK|    4140.0|   Mid|
|  2|  Bob|  Marketing|3200.0|     CZ|    2944.0|Junior|
|  3|Carol|Engineering|5100.0|     SK|    4692.0|Senior|
|  5|  Eve|Engineering|4800.0|     CZ|    4416.0|   Mid|
|  6|Frank|  Marketing|3500.0|     SK|    3220.0|Junior|
|  7|Grace|         HR|3100.0|     SK|    2852.0|Junior|
|  8| Hank|Engineering|5300.0|     CZ|    4876.0|Senior|
+---+-----+-----------+------+-------+----------+------+



In [3]:
# Aggregations
employees.groupBy('department').agg(
    F.count('id').alias('headcount'),
    F.round(F.avg('salary'), 2).alias('avg_salary'),
    F.max('salary').alias('max_salary'),
).orderBy('avg_salary', ascending=False).show()

26/04/07 22:13:57 WARN GlutenFallbackReporter: Validation failed for plan: TakeOrderedAndProject[QueryId=2], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(department#9, Some(Etc/UTC)), class name: ToPrettyString.
26/04/07 22:13:57 WARN TaskSetManager: Lost task 0.0 in stage 5.0 (TID 10) (172.18.0.6 executor 0): org.apache.gluten.exception.GlutenException: org.apache.gluten.exception.GlutenException: Exception: VeloxRuntimeError
Error Source: RUNTIME
Error Code: INVALID_STATE
Reason: Operator::getOutput failed for [operator: TableScan, plan node ID: value-stream:0]: Error during calling Java code from native code: org.apache.gluten.exception.GlutenException: org.apache.gluten.exception.GlutenException: Error during calling Java code from native code: java.lang.UnsupportedOperationException: sun.misc.Unsafe or java.nio.DirectByteBuffer.<init>(long, int) not available
	

Py4JJavaError: An error occurred while calling o105.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 5.0 failed 4 times, most recent failure: Lost task 0.3 in stage 5.0 (TID 13) (172.18.0.5 executor 1): org.apache.gluten.exception.GlutenException: org.apache.gluten.exception.GlutenException: Exception: VeloxRuntimeError
Error Source: RUNTIME
Error Code: INVALID_STATE
Reason: Operator::getOutput failed for [operator: TableScan, plan node ID: value-stream:0]: Error during calling Java code from native code: org.apache.gluten.exception.GlutenException: org.apache.gluten.exception.GlutenException: Error during calling Java code from native code: java.lang.UnsupportedOperationException: sun.misc.Unsafe or java.nio.DirectByteBuffer.<init>(long, int) not available
	at io.netty.util.internal.PlatformDependent.directBuffer(PlatformDependent.java:534)
	at org.apache.gluten.vectorized.LowCopyFileSegmentJniByteInputStream.read(LowCopyFileSegmentJniBy

## 2 — Spark SQL: Window Functions & CTEs

In [4]:
employees.createOrReplaceTempView('employees')

spark.sql('''
  SELECT name, department, salary,
         RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS dept_rank,
         ROUND(salary / AVG(salary) OVER (PARTITION BY department), 2) AS vs_dept_avg
  FROM employees
  ORDER BY department, dept_rank
''').show()

26/04/07 22:15:03 WARN GlutenFallbackReporter: Validation failed for plan: TakeOrderedAndProject[QueryId=4], due to: 
 - Validation failed with exception from: ProjectExecTransformer, reason: Not supported to map spark function name to substrait function name: toprettystring(name#8, Some(Etc/UTC)), class name: ToPrettyString.
26/04/07 22:15:03 WARN TaskSetManager: Lost task 0.0 in stage 8.0 (TID 18) (172.18.0.5 executor 1): org.apache.gluten.exception.GlutenException: org.apache.gluten.exception.GlutenException: Exception: VeloxRuntimeError
Error Source: RUNTIME
Error Code: INVALID_STATE
Reason: Operator::getOutput failed for [operator: TableScan, plan node ID: value-stream:0]: Error during calling Java code from native code: org.apache.gluten.exception.GlutenException: org.apache.gluten.exception.GlutenException: Error during calling Java code from native code: java.lang.UnsupportedOperationException: sun.misc.Unsafe or java.nio.DirectByteBuffer.<init>(long, int) not available
	at io.

Py4JJavaError: An error occurred while calling o111.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 8.0 failed 4 times, most recent failure: Lost task 0.3 in stage 8.0 (TID 21) (172.18.0.5 executor 1): org.apache.gluten.exception.GlutenException: org.apache.gluten.exception.GlutenException: Exception: VeloxRuntimeError
Error Source: RUNTIME
Error Code: INVALID_STATE
Reason: Operator::getOutput failed for [operator: TableScan, plan node ID: value-stream:0]: Error during calling Java code from native code: org.apache.gluten.exception.GlutenException: org.apache.gluten.exception.GlutenException: Error during calling Java code from native code: java.lang.UnsupportedOperationException: sun.misc.Unsafe or java.nio.DirectByteBuffer.<init>(long, int) not available
	at io.netty.util.internal.PlatformDependent.directBuffer(PlatformDependent.java:534)
	at org.apache.gluten.vectorized.LowCopyFileSegmentJniByteInputStream.read(LowCopyFileSegmentJniBy

In [ ]:
spark.sql('''
  WITH dept_stats AS (
    SELECT department, AVG(salary) AS avg_sal, COUNT(*) AS headcount
    FROM employees GROUP BY department
  )
  SELECT e.name, e.salary, ROUND(d.avg_sal,2) AS dept_avg, d.headcount,
         CASE WHEN e.salary > d.avg_sal THEN "above avg" ELSE "below avg" END AS vs_avg
  FROM employees e JOIN dept_stats d USING (department)
  ORDER BY e.department, e.salary DESC
''').show()

## 3 — Caching & Persist

In [ ]:
import time
from pyspark import StorageLevel

big_df = spark.range(1_000_000) \
    .withColumn('value',    (F.col('id') * 3.14).cast('double')) \
    .withColumn('category', (F.col('id') % 10).cast('string'))

t0 = time.time()
big_df.filter(F.col('value') > 1000).count()
big_df.groupBy('category').count().collect()
print(f'Without cache : {time.time()-t0:.2f}s')

big_df.cache()
big_df.count()   # materialise

t1 = time.time()
big_df.filter(F.col('value') > 1000).count()
big_df.groupBy('category').count().collect()
print(f'With cache    : {time.time()-t1:.2f}s')

big_df.unpersist()
print('Cache released.')

## 4 — Partitioning

In [ ]:
df = spark.range(500_000).withColumn('group', (F.col('id') % 5).cast('string'))
print(f'Default        : {df.rdd.getNumPartitions()}')
print(f'repartition(10): {df.repartition(10).rdd.getNumPartitions()}')
print(f'repartition(by col): {df.repartition(5, F.col("group")).rdd.getNumPartitions()}')
print(f'coalesce(4)    : {df.repartition(10).coalesce(4).rdd.getNumPartitions()}')

## 5 — Broadcast Join

In [ ]:
transactions = spark.range(500_000) \
    .withColumn('country_code', (F.col('id') % 3).cast('string')) \
    .withColumn('amount',       (F.col('id') * 1.5).cast('double'))

countries = spark.createDataFrame(
    [('0','Slovakia','EUR'),('1','Czechia','CZK'),('2','Hungary','HUF')],
    ['country_code','country_name','currency']
)

print('=== WITHOUT BROADCAST ===')
transactions.join(countries, 'country_code').explain()

print('=== WITH BROADCAST ===')
transactions.join(F.broadcast(countries), 'country_code').explain()

## 6 — Parquet + Partition Pruning

In [ ]:
employees.write.mode('overwrite').partitionBy('country') \
    .parquet(f'{DATA_DIR}/parquet/employees')

df_sk = spark.read.parquet(f'{DATA_DIR}/parquet/employees') \
    .filter(F.col('country') == 'SK')

df_sk.show()
df_sk.explain()   # look for 'PartitionFilters' in the plan

## 7 — Apache Iceberg: ACID · Time Travel · Schema Evolution

In [ ]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS local.demo')
employees.writeTo('local.demo.employees') \
    .tableProperty('format-version', '2') \
    .partitionedBy('country').createOrReplace()
spark.sql('SELECT * FROM local.demo.employees').show()

In [ ]:
spark.sql("UPDATE local.demo.employees SET salary = salary * 1.1 WHERE department = 'Engineering'")
spark.sql("DELETE FROM local.demo.employees WHERE country = 'HU'")

updates = spark.createDataFrame(
    [(1,'Alice','Engineering',5200.0,'SK'),(9,'Ivan','Finance',4100.0,'SK')],
    ['id','name','department','salary','country']
)
updates.createOrReplaceTempView('updates')
spark.sql('''
  MERGE INTO local.demo.employees t USING updates u ON t.id = u.id
  WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *
''')
spark.sql('SELECT * FROM local.demo.employees ORDER BY id').show()

In [ ]:
# Time travel
spark.sql('SELECT snapshot_id, committed_at, operation FROM local.demo.employees.history').show(truncate=False)

snap = spark.sql(
    'SELECT snapshot_id FROM local.demo.employees.history ORDER BY committed_at LIMIT 1'
).collect()[0][0]
print(f'Original snapshot {snap}:')
spark.sql(f'SELECT * FROM local.demo.employees VERSION AS OF {snap}').show()

# Schema evolution
spark.sql('ALTER TABLE local.demo.employees ADD COLUMN bonus DOUBLE')
spark.sql("UPDATE local.demo.employees SET bonus = salary * 0.1 WHERE department = 'Engineering'")
spark.sql('SELECT id, name, salary, bonus FROM local.demo.employees ORDER BY id').show()

## 8 — Delta Lake: ACID · Time Travel · OPTIMIZE

In [ ]:
employees.write.format('delta').mode('overwrite').partitionBy('country') \
    .save(f'{DATA_DIR}/delta/employees')
spark.sql(f"CREATE TABLE IF NOT EXISTS delta_employees USING delta LOCATION '{DATA_DIR}/delta/employees'")

spark.sql("UPDATE delta_employees SET salary = salary * 1.05 WHERE country = 'SK'")
spark.sql("DELETE FROM delta_employees WHERE department = 'HR'")

spark.sql('DESCRIBE HISTORY delta_employees').select('version','timestamp','operation').show(truncate=False)

print('Version 0:')
spark.read.format('delta').option('versionAsOf', 0).load(f'{DATA_DIR}/delta/employees').show()

spark.sql('OPTIMIZE delta_employees ZORDER BY (id)')
print('OPTIMIZE done.')

## 9 — Structured Streaming (rate source)

In [ ]:
stream_df = spark.readStream.format('rate').option('rowsPerSecond', 20).load()

windowed = stream_df \
    .withWatermark('timestamp', '10 seconds') \
    .groupBy(F.window('timestamp', '5 seconds')) \
    .count()

query = windowed.writeStream.format('memory') \
    .queryName('stream_counts').outputMode('complete').start()

time.sleep(15)
spark.sql('SELECT * FROM stream_counts ORDER BY window').show(truncate=False)
query.stop()
print('Stream stopped.')

## 10 — UDF & Pandas UDF

In [ ]:
import pandas as pd
from pyspark.sql.functions import udf, pandas_udf

@udf(returnType=StringType())
def salary_band(salary):
    if salary >= 5000:   return 'Band A'
    elif salary >= 4000: return 'Band B'
    elif salary >= 3000: return 'Band C'
    else:                return 'Band D'

@pandas_udf(DoubleType())
def tax_estimate(s: pd.Series) -> pd.Series:
    return s.apply(lambda x: x * 0.25 if x > 4000 else x * 0.19)

employees \
    .withColumn('band', salary_band(F.col('salary'))) \
    .withColumn('tax',  tax_estimate(F.col('salary'))) \
    .show()

## 11 — AQE & explain()

In [ ]:
print('AQE enabled:', spark.conf.get('spark.sql.adaptive.enabled'))

# Skewed join — AQE splits the hot partition automatically
skewed = spark.range(1_000_000) \
    .withColumn('key', F.when(F.col('id') < 900_000, F.lit('A')).otherwise((F.col('id') % 100).cast('string'))) \
    .withColumn('value', F.col('id').cast('double'))

small = spark.createDataFrame(
    [(str(i), f'label_{i}') for i in range(100)] + [('A', 'label_A')],
    ['key', 'label']
)

result = skewed.join(small, 'key').groupBy('label').count()
result.explain('formatted')   # look for 'AQE' / 'SkewJoin'
result.show(5)

---
# Part B — Vanilla vs Gluten/Velox Benchmark

Run this notebook **twice**:
1. `docker compose up`  → results saved as `results_vanilla.json`
2. `GLUTEN_ENABLED=true docker compose up`  → results saved as `results_gluten.json`

The last cell renders a comparison chart once both files exist.

## 12 — Load benchmark data

In [ ]:
from pathlib import Path
import json, datetime

RESULTS_DIR = '/workspace/notebooks/results'
Path(RESULTS_DIR).mkdir(exist_ok=True)
MODE = 'gluten' if GLUTEN_ENABLED else 'vanilla'

lineitem = spark.read.parquet(f'{DATA_DIR}/lineitem.parquet')
orders   = spark.read.parquet(f'{DATA_DIR}/orders.parquet')
customer = spark.read.parquet(f'{DATA_DIR}/customer.parquet')

lineitem.createOrReplaceTempView('lineitem')
orders.createOrReplaceTempView('orders')
customer.createOrReplaceTempView('customer')

print(f'Mode     : {MODE}')
print(f'lineitem : {lineitem.count():,}')
print(f'orders   : {orders.count():,}')
print(f'customer : {customer.count():,}')


## 13 — Benchmark queries & runner

In [ ]:
import json, datetime

QUERIES = {
    # Q1 — heavy aggregation on large table (vectorised sum/avg)
    'Q1_heavy_agg': '''
        SELECT l_returnflag, l_linestatus,
               SUM(l_quantity)                                  AS sum_qty,
               SUM(l_extendedprice)                             AS sum_base_price,
               SUM(l_extendedprice * (1 - l_discount))          AS sum_disc_price,
               SUM(l_extendedprice * (1-l_discount)*(1+l_tax))  AS sum_charge,
               AVG(l_quantity)                                  AS avg_qty,
               AVG(l_extendedprice)                             AS avg_price,
               AVG(l_discount)                                  AS avg_disc,
               COUNT(*)                                         AS count_order
        FROM lineitem
        WHERE l_shipdate <= '1998-09-01'
        GROUP BY l_returnflag, l_linestatus
        ORDER BY l_returnflag, l_linestatus
    ''',

    # Q3 — three-table join + aggregation
    'Q3_join_agg': '''
        SELECT l.l_orderkey,
               SUM(l.l_extendedprice * (1 - l.l_discount)) AS revenue,
               o.o_orderdate, o.o_shippriority
        FROM customer c
        JOIN orders   o ON c.c_custkey = o.o_custkey
        JOIN lineitem l ON o.o_orderkey = l.l_orderkey
        WHERE c.c_mktsegment = 'BUILDING'
          AND o.o_orderdate  < '1995-03-15'
          AND l.l_shipdate   > '1995-03-15'
        GROUP BY l.l_orderkey, o.o_orderdate, o.o_shippriority
        ORDER BY revenue DESC, o.o_orderdate
        LIMIT 10
    ''',

    # Q6 — filter + scalar aggregation (lots of reads, minimal shuffle)
    'Q6_filter_scalar': '''
        SELECT SUM(l_extendedprice * l_discount) AS revenue
        FROM lineitem
        WHERE l_shipdate >= '1994-01-01'
          AND l_shipdate <  '1995-01-01'
          AND l_discount BETWEEN 0.05 AND 0.07
          AND l_quantity < 24
    ''',

    # Q12 — join + conditional COUNT
    'Q12_join_count': '''
        SELECT l.l_shipmode,
               SUM(CASE WHEN o.o_orderpriority IN ('1-URGENT','2-HIGH') THEN 1 ELSE 0 END) AS high_line_count,
               SUM(CASE WHEN o.o_orderpriority NOT IN ('1-URGENT','2-HIGH') THEN 1 ELSE 0 END) AS low_line_count
        FROM orders o JOIN lineitem l ON o.o_orderkey = l.l_orderkey
        WHERE l.l_shipmode IN ('MAIL','SHIP')
          AND l.l_commitdate < l.l_shipdate
          AND l.l_shipdate >= '1994-01-01'
          AND l.l_shipdate <  '1995-01-01'
        GROUP BY l.l_shipmode
        ORDER BY l.l_shipmode
    ''',

    # Q_window — window functions (Gluten has dedicated Velox support)
    'Q_window': '''
        SELECT o_custkey, o_orderdate, o_totalprice,
               SUM(o_totalprice) OVER (
                   PARTITION BY o_custkey ORDER BY o_orderdate
                   ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
               ) AS running_3_total,
               RANK() OVER (PARTITION BY o_custkey ORDER BY o_totalprice DESC) AS price_rank
        FROM orders
        WHERE o_orderstatus = 'O'
        LIMIT 50000
    ''',
}

REPEATS      = 3
WARMUP_QUERY = 'Q6_filter_scalar'

def run_query(name: str, sql: str, repeats: int = REPEATS) -> dict:
    times = []
    for i in range(repeats):
        t0 = time.perf_counter()
        cnt = spark.sql(sql).count()
        elapsed = time.perf_counter() - t0
        times.append(elapsed)
        print(f'  run {i+1}/{repeats}: {elapsed:.2f}s  (rows={cnt})')
    best = min(times)
    avg  = sum(times) / len(times)
    print(f'  → best={best:.2f}s  avg={avg:.2f}s')
    return {
        'query': name, 'mode': MODE,
        'best': best, 'avg': avg, 'times': times,
        'ts': datetime.datetime.now().isoformat()
    }

print('Runner ready.')

## 14 — Warmup

In [ ]:
print('Warming up JIT + file metadata cache...')
spark.sql(QUERIES[WARMUP_QUERY]).count()
print('Warmup done.')

## 15 — Run all queries

In [ ]:
results = []
for name, sql in QUERIES.items():
    print(f'\n=== {name} ({MODE}) ===')
    results.append(run_query(name, sql))

out_file = f'{RESULTS_DIR}/results_{MODE}.json'
with open(out_file, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved: {out_file}')

## 16 — Verify Gluten operator offloading

In [ ]:
if GLUTEN_ENABLED:
    print('Physical plan for Q1 — look for "Velox" prefixed operators:')
    spark.sql(QUERIES['Q1_heavy_agg']).explain(extended=False)
    print()
    print('If you see VeloxHashAggregateExec, VeloxProjectExec etc. → Gluten is working.')
    print('If you see only HashAggregateExec → Gluten is falling back to JVM (expected on Spark 4.x with Gluten 1.4).')
else:
    print('Run with GLUTEN_ENABLED=true to check Velox operator offloading.')

## 17 — Compare results (run after both vanilla and gluten passes)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

vanilla_file = f'{RESULTS_DIR}/results_vanilla.json'
gluten_file  = f'{RESULTS_DIR}/results_gluten.json'

if not (Path(vanilla_file).exists() and Path(gluten_file).exists()):
    missing = [f for f in [vanilla_file, gluten_file] if not Path(f).exists()]
    print('Missing result files:', missing)
    print('Run the benchmark once in vanilla mode and once with GLUTEN_ENABLED=true.')
else:
    with open(vanilla_file) as f: v = {r['query']: r for r in json.load(f)}
    with open(gluten_file)  as f: g = {r['query']: r for r in json.load(f)}

    queries  = list(v.keys())
    v_times  = [v[q]['best'] for q in queries]
    g_times  = [g[q]['best'] for q in queries]
    speedups = [vt/gt for vt, gt in zip(v_times, g_times)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Vanilla Spark 4.1.1 vs Gluten/Velox — Local Benchmark', fontsize=13)

    x = np.arange(len(queries))
    w = 0.35
    ax1.bar(x - w/2, v_times, w, label='Vanilla',      color='#D85A30', alpha=0.85)
    ax1.bar(x + w/2, g_times, w, label='Gluten/Velox', color='#1D9E75', alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(queries, rotation=20, ha='right', fontsize=9)
    ax1.set_ylabel('Time (s) — best of 3')
    ax1.set_title('Execution time (lower = better)')
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)

    colors = ['#1D9E75' if s >= 1 else '#D85A30' for s in speedups]
    bars = ax2.bar(queries, speedups, color=colors, alpha=0.85)
    ax2.axhline(1.0, color='gray', linestyle='--', linewidth=0.8)
    ax2.set_xticklabels(queries, rotation=20, ha='right', fontsize=9)
    ax2.set_ylabel('Speedup (Vanilla time / Gluten time)')
    ax2.set_title('Speedup Gluten/Velox vs Vanilla  (>1.0 = Gluten faster)')
    for bar, s in zip(bars, speedups):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{s:.2f}×', ha='center', va='bottom', fontsize=9)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    chart_path = f'{RESULTS_DIR}/benchmark_chart.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\nChart saved: {chart_path}')
    print(f'\n{"Query":<22} {"Vanilla":>9} {"Gluten":>9} {"Speedup":>9}')
    print('-' * 54)
    for q, vt, gt, s in zip(queries, v_times, g_times, speedups):
        print(f'{q:<22} {vt:>8.2f}s {gt:>8.2f}s {s:>8.2f}×')
    print(f'\nAverage speedup: {sum(speedups)/len(speedups):.2f}×')